# 02 · Linear regression (MOS)

**Ne yapıyoruz?** GFS yarın için bir rüzgâr hızı tahmin ediyor ama LTBJ'de sistematik olarak düşük kalıyor.
Amacımız GFS'in tahminini **girdi** olarak alıp **gerçekte ölçülecek rüzgârı** tahmin eden küçük bir model kurmak.
Meteorolojide buna MOS (Model Output Statistics) deniyor.

**Geçmesi gereken çizgi:** 01'deki en iyi yöntem, yani aylık bias düzeltmesi, test MAE **3,65 knot** verdi.

Her hücreyi sırayla çalıştır (Shift+Enter). Önce üstündeki açıklamayı oku.

## 0) Veriyi yükle

`load_aligned()` 01'deki işi yapıyor: her GFS saatine en yakın METAR ölçümünü eşliyor.
`split()` veriyi zamana göre bölüyor: 1 Kasım 2025'ten önceki veri **train** (model bundan öğrenir), sonrası **test** (model bunu hiç görmez, sınavı burada yapar).

Neden rastgele değil de zamana göre bölüyoruz? Gerçekte model hep **geleceği** tahmin edecek.
Rastgele bölseydik model, test'teki bir saatin bir saat öncesini ve sonrasını train'de görmüş olurdu. Bu kopya çekmek gibi olur, sonuç gerçekte olduğundan iyi çıkar.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

from ruzgar.data import load_aligned, split

aligned = load_aligned()
train, test = split(aligned)
print("train:", len(train), "satır   test:", len(test), "satır")
aligned[["time", "speed_kn_gfs", "speed_kn_obs"]].head()

train: 14629 satır   test: 8804 satır


,time,speed_kn_gfs,speed_kn_obs
0,2024-03-01 01:00:00+00:00,1.8,2.0
1,2024-03-01 02:00:00+00:00,2.0,5.0
2,2024-03-01 03:00:00+00:00,1.6,2.0
3,2024-03-01 04:00:00+00:00,0.4,6.0
4,2024-03-01 05:00:00+00:00,0.9,4.0


## 1) Tek girdili model: `ölçüm = a · GFS + b`

**Fikir:** 01'deki sabit düzeltme aslında şuydu: `ölçüm = 1 · GFS + 3,7`. Yani eğim 1'e sabitti, biz yalnız sabiti bulduk.
Linear regression hem **eğimi (a)** hem **sabiti (b)** veriden kendisi bulur. Bunu yaparken train'deki hataların karelerinin toplamını en küçük yapan doğruyu seçer.

**sklearn'de her model aynı üç adımla çalışır:**
1. `X`: girdiler. Her satır bir saat, her sütun bir **feature** (özellik). sklearn bunun bir tablo olmasını ister, bu yüzden çift köşeli parantez kullanıyoruz: `[["speed_kn_gfs"]]`.
2. `y`: tahmin etmek istediğimiz şey, yani ölçülen hız.
3. `model.fit(X, y)` öğrenir, `model.predict(X)` tahmin üretir.

Sonra test'teki tahminlerle gerçek ölçüm arasındaki ortalama mutlak farka bakıyoruz (MAE).

In [ ]:
# Girdiler (X) ve hedef (y)
X_train = train[["speed_kn_gfs"]]
y_train = train["speed_kn_obs"]
X_test = test[["speed_kn_gfs"]]
y_test = test["speed_kn_obs"]

# Öğren (yalnız train'den!)
model = LinearRegression()
model.fit(X_train, y_train)

# Öğrendiği doğru
a = model.coef_[0]      # eğim
b = model.intercept_    # sabit
print(f"Öğrenilen: ölçüm = {a:.2f} · GFS + {b:.2f}")

# Test'te sına
tahmin = model.predict(X_test)
mae_tek = mean_absolute_error(y_test, tahmin)
print(f"Test MAE: {mae_tek:.2f} knot   (ham GFS 4,41 · sabit düzeltme 3,80 · aylık 3,65)")

Öğrenilen: ölçüm = 1.10 · GFS + 3.27
Test MAE: 3.77 knot   (ham GFS 4,41 · sabit düzeltme 3,80 · aylık 3,65)


**Nasıl okunur?** Eğim 1'den büyük çıktı (~1,10). Yani GFS yalnız sabit bir miktar düşük kalmıyor, rüzgâr kuvvetlendikçe açık da büyüyor.
GFS 5 kn dediğinde düzeltme yaklaşık +3,8 kn oluyor, 20 kn dediğinde +5,2 kn.

MAE sabit düzeltmeye göre biraz iyileşti, ama aylık düzeltmenin (3,65) hâlâ gerisinde. Çünkü model saati ve mevsimi hiç bilmiyor.

✏️ **Alıştırma:** Aşağıdaki hücrede GFS'e 5, 10 ve 20 değerlerini verip modelin ne dediğine bak. Elle `a·x + b` hesaplayıp karşılaştır.

In [ ]:
ornek = pd.DataFrame({"speed_kn_gfs": [5, 10, 20]})
model.predict(ornek)

array([ 8.77009649, 14.26818501, 25.26436206])

## 1b) `fit` aslında ne yapıyor? Elle çöz (bu hücreyi sen yazıyorsun)

`model.fit` bir kara kutu değil. Tek girdili linear regression şu soruyu soruyor:
**hangi `a` ve `b` için `a·GFS + b` ile ölçüm arasındaki farkların kareleri toplamı en küçük olur?**
Bu sorunun kesin bir cevabı var, deneme yanılma gerekmiyor. Bir matris denklemi çözülüyor, adı **en küçük kareler** (least squares).

**Hile: sabiti de bir sütun gibi yazmak.** Her satır için tahmin şu:

`tahmin = b · 1 + a · GFS`

Yani başa her satırda değeri **1** olan bir sütun koyarsak sabit `b` de sıradan bir katsayıya dönüşür:

| 1 | GFS |
|---|---|
| 1 | 1.8 |
| 1 | 2.0 |
| … | … |

Bu tabloya `X` dersek numpy `X · w ≈ y` denklemini en küçük karelerle çözer ve `w = [b, a]` verir.

**Gereken kalıplar:**
- pandas sütununu numpy dizisine çevirmek: `seri.to_numpy()`
- n tane 1: `np.ones(n)`, satır sayısı: `len(train)`
- iki diziyi yan yana sütun yapmak: `np.column_stack([dizi1, dizi2])`
- şekli kontrol etmek: `X_elle.shape` ifadesi `(14629, 2)` olmalı
- çözmek: `np.linalg.lstsq(X_elle, y, rcond=None)`. Bu bir demet (tuple) döndürür, ilk elemanı `w`. Yazımı: `w = np.linalg.lstsq(...)[0]`

**Kontrol:** sklearn'ün bulduğu **a = 1,10** ve **b = 3,27** ile aynı sayıları bulmalısın. Takılırsan bana yaz, önce ipucu veririm.

In [ ]:
# 1) train'deki GFS hızını numpy dizisine çevir → x
x = train["speed_kn_gfs"].to_numpy()
# 2) Başına 1'lerden oluşan bir sütun ekle → X_elle (şekli (satır sayısı, 2) olmalı)
X_elle = np.column_stack([np.ones(len(train)),x])
print(X_elle.shape)
# 3) Hedef: train'deki ölçülen hız, numpy dizisi olarak → y
y = train["speed_kn_obs"].to_numpy()
# 4) lstsq ile çöz → w
w = np.linalg.lstsq(X_elle, y, rcond=None)[0]
# 5) Yazdır: b = w[0], a = w[1]
b = w[0] 
a = w[1]
print(f"a = {a:.2f}, b = {b:.2f}")

(14629, 2)
a = 1.10, b = 3.27


**Meraklısına (isteğe bağlı):** aynı sonucu formülle de bulabilirsin. En küçük kareler çözümü `w = (XᵀX)⁻¹ Xᵀ y`. numpy'da `@` matris çarpımı demek, `.T` de transpoz:
`np.linalg.solve(X_elle.T @ X_elle, X_elle.T @ y)`. Sonuç aynı çıkmalı. Pratikte `lstsq` tercih edilir, çünkü sayısal olarak daha kararlı.

## 2) Saati ekle

01'de hatanın saate göre değiştiğini gördük. Modele saati nasıl vereceğiz?

**Saati düz sayı (0–23) olarak vermek yanlış olur:** linear regression her feature için tek bir çarpan öğrenir. O zaman "saat 1 arttıkça rüzgâr hep şu kadar artar" demiş olur.
Ayrıca 23 ile 0 aslında komşu saatler, ama sayı olarak birbirinden en uzak değerler.

**Çözüm: saati bir saat kadranına yerleştir.** Her saati bir çember üzerinde bir nokta yap ve noktanın iki koordinatını ver:
- `saat_sin = sin(2π · saat / 24)`
- `saat_cos = cos(2π · saat / 24)`

Böylece 23 ile 0 çemberde yan yana düşer. Model bu iki sayıyla "günün şu saatlerinde fazla, şu saatlerinde az" şeklinde tek bir dalga öğrenebilir.

Aynı dönüşümü hem train hem test'e uygulamamız gerekiyor. Tekrar yazmamak için bir fonksiyon kullanıyoruz.

In [ ]:
def ozellikler(df):
    saat = df["time"].dt.hour
    X = pd.DataFrame(index=df.index)
    X["gfs"] = df["speed_kn_gfs"]
    X["saat_sin"] = np.sin(2 * np.pi * saat / 24)
    X["saat_cos"] = np.cos(2 * np.pi * saat / 24)
    return X

ozellikler(train).head()

,gfs,saat_sin,saat_cos
0,1.8,0.258819,0.965926
1,2.0,0.500000,0.866025
2,1.6,0.707107,0.707107
3,0.4,0.866025,0.500000
4,0.9,0.965926,0.258819


In [ ]:
model = LinearRegression().fit(ozellikler(train), y_train)
mae_saat = mean_absolute_error(y_test, model.predict(ozellikler(test)))
print(f"Test MAE: {mae_saat:.2f}   (önceki: {mae_tek:.2f})")

Test MAE: 3.76   (önceki: 3.77)


**Neredeyse hiç değişmedi. Neden?**

01'de saatlik hatanın büyük kısmı **yazın** ortaya çıkıyordu (imbat). Kışın saatin etkisi çok küçük.
Bu model ise saatin etkisini bütün yıl için **tek bir dalga** olarak öğreniyor. Yaz ile kışın ortalaması alınınca etki siliniyor.

Bu, linear regression'ın temel sınırı: feature'ların etkisini **ayrı ayrı toplar**, "yaz **ve** öğleden sonra" gibi birleşimleri kendiliğinden göremez.
Önce mevsimi de ekleyelim, sonra bu sınırı nasıl aşacağımıza bakarız.

## 3) Mevsimi (ayı) ekle

Ay da saat gibi daireseldir: Aralık (12) ile Ocak (1) komşudur. Aynı kadran yöntemini 12 ile kullanıyoruz.

✏️ **Alıştırma:** Aşağıdaki fonksiyonda `ay_sin` satırı hazır. `ay_cos` satırını sen ekle (saat_cos'a bak, 24 yerine 12 kullan), sonra iki hücreyi çalıştır.

**Kontrol:** satır yokken test MAE 3,70 çıkıyor, doğru yazınca **3,55** çıkmalı.
(Tabloda `6.1e-17` gibi bir sayı görürsen o aslında 0: Mart için cos(90°) = 0, bilgisayar ondalıklı sayıda tam 0 tutturamıyor.)

In [ ]:
def ozellikler(df):
    saat = df["time"].dt.hour
    ay = df["time"].dt.month
    X = pd.DataFrame(index=df.index)
    X["gfs"] = df["speed_kn_gfs"]
    X["saat_sin"] = np.sin(2 * np.pi * saat / 24)
    X["saat_cos"] = np.cos(2 * np.pi * saat / 24)
    X["ay_sin"] = np.sin(2 * np.pi * ay / 12)
    X["ay_cos"] = np.cos(2 * np.pi * ay / 12)
    return X

ozellikler(train).head()

,gfs,saat_sin,saat_cos,ay_sin,ay_cos
0,1.8,0.258819,0.965926,1.0,6.123234e-17
1,2.0,0.500000,0.866025,1.0,6.123234e-17
2,1.6,0.707107,0.707107,1.0,6.123234e-17
3,0.4,0.866025,0.500000,1.0,6.123234e-17
4,0.9,0.965926,0.258819,1.0,6.123234e-17


In [ ]:
model = LinearRegression().fit(ozellikler(train), y_train)
mae_ay = mean_absolute_error(y_test, model.predict(ozellikler(test)))
print(f"Test MAE: {mae_ay:.2f}   (hedef: 3,65'in altı)")

Test MAE: 3.55   (hedef: 3,65'in altı)


## 4) Rüzgâr yönünü ekle (bu hücreyi sen yazıyorsun)

**Fikir:** imbat batıdan, poyraz kuzeyden eser. GFS'in hatası yöne göre değişiyor olabilir.
Yön de saat gibi daireseldir: 350° ile 10° komşudur. Bu yüzden derece değerini düz sayı olarak vermiyoruz.

**Bu sefer sin/cos'u hızla çarpıyoruz.** Rüzgârı bir ok gibi düşün: ok, yönü gösteriyor, uzunluğu da hız.
Okun doğu-batı bileşeni `u`, kuzey-güney bileşeni `v`:
- `u = hız · sin(yön)`
- `v = hız · cos(yön)`

Hızla çarpmanın bir faydası var: 1 knotluk rüzgârın yönü pek önemli değil, 20 knotluk batı rüzgârı ise önemli. Çarpınca model ikisini ayırt edebiliyor.
(Meteorolojide işaretin tersi kullanılır. Linear regression'da bu önemli değil, model katsayının işaretini kendisi çevirir.)

**Gereken kalıplar:**
- Yön sütunu: `df["dir_deg_gfs"]` (derece cinsinden, GFS'in tahmini)
- `np.sin` radyan bekler, dereceyi radyana çevirmek için: `np.deg2rad(derece)`
- Hız sütunu: `df["speed_kn_gfs"]`

**Beklenen:** doğru yazarsan test MAE yaklaşık **3,1** çıkmalı. Takılırsan bana yaz, önce ipucu veririm.

In [ ]:
def ozellikler(df):
    saat = df["time"].dt.hour
    ay = df["time"].dt.month
    X = pd.DataFrame(index=df.index)
    X["gfs"] = df["speed_kn_gfs"]
    X["saat_sin"] = np.sin(2 * np.pi * saat / 24)
    X["saat_cos"] = np.cos(2 * np.pi * saat / 24)
    X["ay_sin"] = np.sin(2 * np.pi * ay / 12)
    X["ay_cos"] = np.cos(2 * np.pi * ay / 12)
    # 1) GFS'in yönünü radyana çevir ve bir değişkene ata
    direction = df["dir_deg_gfs"]
    direction_rad = np.deg2rad(direction)
    # 2) u_ruzgar sütununu ekle: hız · sin(yön)
    X["u_ruzgar"] = df["speed_kn_gfs"] * np.sin(direction_rad)
    # 3) v_ruzgar sütununu ekle: hız · cos(yön)
    X["v_ruzgar"] = df["speed_kn_gfs"] * np.cos(direction_rad)

    return X

ozellikler(train).head()

,gfs,saat_sin,saat_cos,ay_sin,ay_cos,u_ruzgar,v_ruzgar
0,1.8,0.258819,0.965926,1.0,6.123234e-17,1.180906,-1.358477
1,2.0,0.500000,0.866025,1.0,6.123234e-17,-1.912610,0.584743
2,1.6,0.707107,0.707107,1.0,6.123234e-17,-0.387075,1.552473
3,0.4,0.866025,0.500000,1.0,6.123234e-17,-0.356403,0.181596
4,0.9,0.965926,0.258819,1.0,6.123234e-17,-0.408591,-0.801906


In [ ]:
# Bu hücre hazır: önceki adımlarla aynı, değiştirme
model = LinearRegression().fit(ozellikler(train), y_train)
mae_yon = mean_absolute_error(y_test, model.predict(ozellikler(test)))
print(f"Test MAE: {mae_yon:.2f}   (önceki: {mae_ay:.2f})")

Test MAE: 3.14   (önceki: 3.55)


## Özet tablosu

Her denemede buraya bir satır ekleyeceğiz.

In [ ]:
pd.DataFrame({
    "yöntem": ["ham GFS", "sabit düzeltme", "aylık düzeltme", "LR: gfs", "LR: + saat", "LR: + ay", "LR: + yön (u/v)"],
    "test MAE": [4.41, 3.80, 3.65, mae_tek, mae_saat, mae_ay, mae_yon],
}).round(2)

,yöntem,test MAE
0,ham GFS,4.41
1,sabit düzeltme,3.80
2,aylık düzeltme,3.65
3,LR: gfs,3.77
4,LR: + saat,3.76
5,LR: + ay,3.55
6,LR: + yön (u/v),3.14
